# Suffix Automaton + Dynamic-Length Speculative Decoding

**Weekend project** — reimplements the core ideas from Baseten's `sa_spec` repo and adds **dynamic draft length control**, the explicit future-work item from their [SA MTP blog post (Jan 27, 2026)](https://www.baseten.co/blog/sa-mtp).

### Five decoding modes
| Mode | Description |
|------|-------------|
| `autoregressive` | Baseline: one target step per token |
| `specdec` | Draft model only (fixed draft length) |
| `sa_only` | Suffix automaton only (fixed draft length) |
| `hybrid_fixed` | SA + draft model, fixed length |
| `hybrid_dynamic` | SA + draft model, **adaptive draft length** (novel) |

**Runtime**: T4 GPU on Google Colab (free tier)

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets numpy matplotlib tqdm accelerate

In [ ]:
import os, sys

REPO_DIR = "BasetenResearch_Project"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Matthew-Matta/BasetenResearch_Project.git

# chdir first, then insert absolute cwd — avoids double-nested path bug
os.chdir(REPO_DIR)
repo_abs = os.getcwd()   # e.g. /content/BasetenResearch_Project
if repo_abs not in sys.path:
    sys.path.insert(0, repo_abs)

print("Working directory:", os.getcwd())
print("sys.path[0]:", sys.path[0])

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd

from transformers import AutoModelForCausalLM, AutoTokenizer

from src.suffix_automaton import SuffixAutomaton, DualSuffixAutomaton
from src.speculative_decode import HybridSpecDecoder
from src.utils import plot_benchmark_results, print_summary_table

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Device: {DEVICE}  |  dtype: {DTYPE}")

## 2. Suffix Automaton — unit test

Build from `[1,2,3,1,2]`, query with `[1,2]` → expect draft `[3]`, match_len=2.

In [ ]:
sa = SuffixAutomaton()
sa.build([1, 2, 3, 1, 2])

drafts, match_len = sa.query([1, 2], max_draft_len=4)
print(f"Draft tokens : {drafts}")
print(f"Match length : {match_len}")
assert match_len == 2, f"Expected match_len=2, got {match_len}"
assert 3 in drafts, f"Expected token 3 in drafts, got {drafts}"
print("✓  Unit test passed")

## 3. Load models

In [ ]:
TARGET_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DRAFT_NAME  = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

print("Loading target model…")
target_tok   = AutoTokenizer.from_pretrained(TARGET_NAME)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_NAME, torch_dtype=DTYPE, device_map="auto"
)

print("Loading draft model…")
draft_tok   = AutoTokenizer.from_pretrained(DRAFT_NAME)
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_NAME, torch_dtype=DTYPE, device_map="auto"
)

decoder = HybridSpecDecoder(
    target_model=target_model,
    target_tokenizer=target_tok,
    draft_model=draft_model,
    draft_tokenizer=draft_tok,
    device=DEVICE,
)
print("Models ready.")

## 4. Single prompt demo — all five modes

In [ ]:
DEMO_PROMPT = (
    "Write a Python function that implements binary search on a sorted list. "
    "Include docstring and type hints."
)

MODES = ["autoregressive", "specdec", "sa_only", "hybrid_fixed", "hybrid_dynamic"]
demo_results = {}

for mode in MODES:
    print(f"\n{'='*60}")
    print(f"Mode: {mode}")
    print('='*60)
    text, metrics = decoder.generate(
        prompt=DEMO_PROMPT,
        max_new_tokens=150,
        mode=mode,
        num_draft_tokens=4,
        sa_threshold=2,   # lowered: SA fires on 2-token context matches
    )
    demo_results[mode] = (text, metrics)
    print(text[:400], "…" if len(text) > 400 else "")
    print(f"\n  TPS={metrics.tokens_per_second:.1f}  "
          f"TTFT={metrics.ttft_s:.3f}s  "
          f"accept={metrics.acceptance_rate:.2%}  "
          f"avg_draft_len={metrics.avg_draft_length:.2f}")

In [ ]:
rows = []
for mode, (_, m) in demo_results.items():
    rows.append({
        "Mode": mode,
        "TPS": f"{m.tokens_per_second:.1f}",
        "TTFT (s)": f"{m.ttft_s:.4f}",
        "Tokens": m.tokens_generated,
        "Accept %": f"{m.acceptance_rate:.1%}",
        "SA acc %": f"{m.sa_acceptance_rate:.1%}",
        "Draft acc %": f"{m.draft_acceptance_rate:.1%}",
        "Avg draft len": f"{m.avg_draft_length:.2f}",
    })
df = pd.DataFrame(rows)
display(df)

## 5. Mini-benchmark — 10 prompts, TPS comparison

In [ ]:
from src.benchmark import load_dataset, run_benchmark

mini_prompts = load_dataset(n_samples=10)
print(f"Loaded {len(mini_prompts)} prompts for mini-benchmark.")

mini_results = {}
for mode in MODES:
    print(f"Running {mode}…")
    mini_results[mode] = run_benchmark(
        decoder=decoder,
        prompts=mini_prompts,
        method=mode,
        max_new_tokens=80,
        num_draft_tokens=4,
        sa_threshold=2,
    )
print("Done.")

**Note on sampled results (temperature=1.0):** With this 3x model ratio (1.5B/0.5B), the draft model's acceptance rate at temperature=1.0 is ~9.9% — not high enough to offset the draft+verification overhead, so `specdec` ends up slower than autoregressive. This is expected behavior: Baseten's production deployments use a 10x ratio (70B/7B) where acceptance rates are much higher. The greedy results in Section 10 show the implementation is correct — at temperature=0, acceptance jumps to ~79% and speculative decoding achieves 3.22x speedup.

In [ ]:
methods = list(mini_results.keys())
mean_tps = [np.mean([m.tokens_per_second for m in mini_results[meth]]) for meth in methods]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, mean_tps, color=plt.cm.tab10(np.linspace(0, 0.9, len(methods))))
ax.set_ylabel("Tokens per Second (TPS)")
ax.set_title("Mini-Benchmark TPS — 10 code prompts")
ax.set_xticklabels(methods, rotation=15, ha="right")
for bar, v in zip(bars, mean_tps):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.2, f"{v:.1f}", ha="center", fontsize=9)
fig.tight_layout()
plt.show()

## 6. Dynamic draft length visualisation

Shows how `hybrid_dynamic` adapts the number of speculative tokens over the course of generation.

In [ ]:
if "hybrid_dynamic" in mini_results and mini_results["hybrid_dynamic"]:
    # Concatenate histories from all 10 prompts
    all_histories = []
    for m in mini_results["hybrid_dynamic"]:
        all_histories.extend(m.draft_length_history)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Left: length over steps
    axes[0].plot(all_histories, linewidth=0.8, alpha=0.8)
    axes[0].axhline(np.mean(all_histories), color="red", linestyle="--",
                    label=f"mean={np.mean(all_histories):.2f}")
    axes[0].set_xlabel("Decoding step (all prompts concatenated)")
    axes[0].set_ylabel("Draft length")
    axes[0].set_title("Adaptive Draft Length — hybrid_dynamic")
    axes[0].legend()

    # Right: histogram
    axes[1].hist(all_histories, bins=range(1, 13), edgecolor="black", color="steelblue", alpha=0.8)
    axes[1].set_xlabel("Draft length")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Draft Length Distribution")

    fig.tight_layout()
    plt.show()
else:
    print("No hybrid_dynamic results available.")

## 7. Acceptance rate breakdown

In [ ]:
print_summary_table(mini_results)

In [ ]:
methods = list(mini_results.keys())
sa_rates      = [np.mean([m.sa_acceptance_rate    for m in mini_results[meth]]) for meth in methods]
draft_rates   = [np.mean([m.draft_acceptance_rate for m in mini_results[meth]]) for meth in methods]
overall_rates = [np.mean([m.acceptance_rate       for m in mini_results[meth]]) for meth in methods]

x = np.arange(len(methods))
width = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width, sa_rates,      width, label="SA",      color="steelblue")
ax.bar(x,         draft_rates,   width, label="Draft",   color="darkorange")
ax.bar(x + width, overall_rates, width, label="Overall", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, ha="right")
ax.set_ylabel("Acceptance Rate")
ax.set_ylim(0, 1.1)
ax.set_title("Acceptance Rate Breakdown by Source")
ax.legend()
fig.tight_layout()
plt.show()

## 8. Save results

## 9. SA Showcase — repetitive code prompt

SA finds matches when generated tokens repeat substrings from the context.
A prompt requiring multiple methods with shared parameter signatures gives SA real work to do.

**Why temperature=1.0?** SA predicts tokens based on what followed a context in the past.
At temperature=1.0 the model sometimes regenerates prompt patterns verbatim (e.g.
`(self, x: float, y: float) -> float:`, `return x`) — those are exactly the substrings
the SA was built from, so it fires reliably. At temperature=0 the model takes a single
deterministic path that often diverges from the prompt's exact phrasing, giving SA fewer
opportunities to match.

Watch `avg_draft_len > 1` and `SA acc %` become non-zero.

In [ ]:
SA_PROMPT = '''Complete this Python Calculator class by implementing the remaining methods
following the same pattern as the examples below:

class Calculator:
    def add(self, x: float, y: float) -> float:
        """Add x and y."""
        return x + y

    def subtract(self, x: float, y: float) -> float:
        """Subtract y from x."""
        return x - y

    def multiply(self, x: float, y: float) -> float:
        # implement this

    def divide(self, x: float, y: float) -> float:
        # implement this, raise ValueError if y == 0

    def power(self, x: float, y: float) -> float:
        # implement this

    def modulo(self, x: float, y: float) -> float:
        # implement this'''

# The SA is built from this prompt.  It contains repeated substrings like:
#   (self, x: float, y: float) -> float:
#   """..."""
#   return x
# At temperature=1.0 the model sometimes regenerates these patterns verbatim,
# giving the SA real match opportunities.
# sa_threshold=2: SA fires when context matches >= 2 tokens.

print('Running SA showcase (temp=1.0, repetitive method signatures -> SA matches)...\n')
sa_showcase = {}
for mode in ['autoregressive', 'sa_only', 'hybrid_dynamic']:
    _, m = decoder.generate(
        prompt=SA_PROMPT,
        max_new_tokens=300,
        mode=mode,
        num_draft_tokens=4,
        sa_threshold=2,
        temperature=1.0,   # SA matches stochastic generation; greedy avoids prompt patterns
    )
    sa_showcase[mode] = m
    print(f'{mode:<20}  TPS={m.tokens_per_second:.1f}  '
          f'SA_acc={m.sa_acceptance_rate:.1%}  '
          f'avg_draft_len={m.avg_draft_length:.2f}  '
          f'source_mix={dict((s, m.source_history.count(s)) for s in set(m.source_history))}')


### DLC Source Adaptation — the novel contribution

Standard DLC adapts draft **length**. This implementation extends it to adapt draft **source** —
routing each step to:
- **SA** (zero cost) when context matches a known pattern
- **Draft model** (when acceptance rate justifies the overhead: `rate × draft_len > 1.0`)
- **AR fallback** (single target step) when neither source is economical

The scatter plot below shows which source was chosen at each step and what draft length was used,
proving the DLC is doing true per-source adaptation.

In [ ]:
if "hybrid_dynamic" in sa_showcase:
    m = sa_showcase["hybrid_dynamic"]
    sources = m.source_history
    lengths = m.draft_length_history

    sa_steps    = [(i, l) for i, (s, l) in enumerate(zip(sources, lengths)) if s == "SA"]
    draft_steps = [(i, l) for i, (s, l) in enumerate(zip(sources, lengths)) if s == "draft"]
    ar_steps    = [(i, l) for i, (s, l) in enumerate(zip(sources, lengths)) if s == "autoregressive"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Left: timeline — which source at each step, and its draft length
    if sa_steps:
        axes[0].scatter(*zip(*sa_steps),    label=f"SA (n={len(sa_steps)})",
                        color="steelblue",  alpha=0.8, s=50)
    if draft_steps:
        axes[0].scatter(*zip(*draft_steps), label=f"draft model (n={len(draft_steps)})",
                        color="darkorange", alpha=0.8, s=50)
    if ar_steps:
        axes[0].scatter(*zip(*ar_steps),    label=f"AR fallback (n={len(ar_steps)})",
                        color="seagreen",   alpha=0.8, s=50, marker="x")
    axes[0].set_xlabel("Decoding step")
    axes[0].set_ylabel("Draft length used")
    axes[0].set_title("DLC: per-source draft length adaptation\n(hybrid_dynamic, SA showcase)")
    axes[0].legend()

    # Right: source distribution pie
    counts = {"SA": len(sa_steps), "draft": len(draft_steps), "AR fallback": len(ar_steps)}
    labels = [f"{k}\n({v} steps)" for k, v in counts.items() if v > 0]
    sizes  = [v for v in counts.values() if v > 0]
    pie_colors = ["steelblue", "darkorange", "seagreen"][:len(sizes)]
    axes[1].pie(sizes, labels=labels, colors=pie_colors, autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Source distribution per step\n(hybrid_dynamic, SA showcase)")

    fig.tight_layout()
    plt.show()

    # Print per-source mean draft lengths
    print("Per-source mean draft length:")
    if sa_steps:
        print(f"  SA:           {np.mean([l for _, l in sa_steps]):.2f}")
    if draft_steps:
        print(f"  Draft model:  {np.mean([l for _, l in draft_steps]):.2f}")
    if ar_steps:
        print(f"  AR fallback:  1.00 (always)")
else:
    print("hybrid_dynamic not in sa_showcase — run the SA showcase cell first.")

## 10. Greedy decoding (temperature=0) — spec decoding wins

With deterministic decoding, acceptance rates jump to 70-80%+ and speculative decoding
clearly outperforms autoregressive. This is the regime where Baseten's production
deployments operate (beam search / greedy constrained generation).

In [ ]:
GREEDY_PROMPT = (
    "Write a Python function that implements binary search on a sorted list. "
    "Include docstring and type hints."
)

print('Greedy decoding comparison (temperature=0):\n')
greedy_results = {}
for mode in ['autoregressive', 'specdec', 'hybrid_dynamic']:
    _, m = decoder.generate(
        prompt=GREEDY_PROMPT,
        max_new_tokens=150,
        mode=mode,
        num_draft_tokens=4,
        sa_threshold=2,
        temperature=0.0,
    )
    greedy_results[mode] = m
    print(f'{mode:<20}  TPS={m.tokens_per_second:.1f}  accept={m.acceptance_rate:.1%}  '
          f'avg_draft_len={m.avg_draft_length:.2f}')

print()
best = max(greedy_results, key=lambda m: greedy_results[m].tokens_per_second)
ar_tps   = greedy_results['autoregressive'].tokens_per_second
best_tps = greedy_results[best].tokens_per_second
print(f'Speedup vs autoregressive: {best_tps/ar_tps:.2f}x  ({best})')


In [ ]:
# Bar chart — greedy decoding
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
methods_g = list(greedy_results.keys())
tps_g = [greedy_results[m].tokens_per_second for m in methods_g]
acc_g = [greedy_results[m].acceptance_rate * 100 for m in methods_g]
colors = ['#2196F3', '#FF9800', '#4CAF50']

axes[0].bar(methods_g, tps_g, color=colors)
axes[0].set_ylabel('TPS')
axes[0].set_title('Throughput — greedy (temperature=0)')
for i, v in enumerate(tps_g):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')

axes[1].bar(methods_g, acc_g, color=colors)
axes[1].set_ylabel('Acceptance Rate %')
axes[1].set_ylim(0, 110)
axes[1].set_title('Acceptance Rate — greedy (temperature=0)')
for i, v in enumerate(acc_g):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

fig.tight_layout()
plt.show()


In [ ]:
import os
from src.benchmark import save_results
from src.utils import plot_benchmark_results

os.makedirs("results/figures", exist_ok=True)
save_results(mini_results, path="results/benchmarks.json")
plot_benchmark_results(mini_results, output_dir="results/figures")
print("All results saved.")